In [1]:
# Reopening the index

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [3]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5)

In [4]:
results[1]['question']
results

[{'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'Error: How to fix requests library only installs v2.28 instead of v2.32 required for lancedb?',
  'answer': 'If you encounter a 401 Client Error, it may indicate the need to grant access to the key or that the key is incorrect.\n\nTo install the correct version directly from the source, use the following command:\n\n```bash\npip install "requests @ https://github.com/psf/requests/archive/refs/tags/v2.32.3.zip"\n```',
  'doc_id': '4b30b918bc'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'dotenv is not recognized. What should I install?',
  'answer': 'Install `python-dotenv`:\n\n```bash\nuv add python-dotenv\n```\n\nThen import and use it in Python:\n\n```python\nfrom dotenv import load_dotenv\n\nload_dotenv()\n```\n\nThe package is documented here: [python-dotenv](https://pypi.org/project/python-dotenv/).',
  'doc_id': '0bed1f48da'}]

In [5]:
# Using sqlitesearch vector search in RAG


In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [3]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [5]:
vs_index.close()